### solving the 1D time-independent Schrodinger equation numerically using the finite-difference method, and comparing to the analytical solution

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def build_hamiltonian(N, L):
    """
    Build the finite-difference Hamiltonian matrix for a 1D particle in a box.

    Parameters
    ----------
    N : int
        Number of interior grid points.
    L : float
        Length of the box.

    Returns
    -------
    H : ndarray, shape (N, N)
        The Hamiltonian matrix (kinetic energy only, since V=0 inside).
    x : ndarray, shape (N,)
        The interior grid points.
    dx : float
        Grid spacing.
    """
    dx = L / (N + 1)
    x = np.linspace(dx, L - dx, N)  # interior points only

    # Kinetic energy operator: T = -1/2 * d^2/dx^2
    # The second derivative finite-difference matrix has:
    #   diagonal:     -2/dx^2
    #   off-diagonal: +1/dx^2
    # Multiplied by -1/2 for the kinetic energy:
    #   diagonal:     +1/dx^2
    #   off-diagonal: -1/(2*dx^2)
    #
    # Wait -- let's be careful. The second-derivative matrix D2 has elements:
    #   D2[j,j]   = -2/dx^2
    #   D2[j,j-1] = D2[j,j+1] = 1/dx^2
    #
    # The Hamiltonian is H = -1/2 * D2, so:
    #   H[j,j]   = -1/2 * (-2/dx^2) = 1/dx^2
    #   H[j,j+-1] = -1/2 * (1/dx^2) = -1/(2*dx^2)

    diag = np.ones(N) / dx**2            # main diagonal
    off_diag = -0.5 * np.ones(N - 1) / dx**2  # super/sub diagonal

    H = np.diag(diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

    return H, x, dx


In [3]:
def analytical_energies(n_levels, L):
    """
    Analytical energy eigenvalues for particle in a box (atomic units).

    E_n = n^2 * pi^2 / (2 * L^2)
    """
    n = np.arange(1, n_levels + 1)
    return n**2 * np.pi**2 / (2 * L**2)


In [4]:
def analytical_wavefunctions(x, n_levels, L):
    """
    Analytical wavefunctions for particle in a box (atomic units).

    psi_n(x) = sqrt(2/L) * sin(n * pi * x / L)

    Returns
    -------
    psi : ndarray, shape (len(x), n_levels)
        Each column is a normalized wavefunction.
    """
    n = np.arange(1, n_levels + 1)
    # Broadcasting: x is (N,1), n is (1,n_levels)
    psi = np.sqrt(2 / L) * np.sin(np.outer(x, n * np.pi / L))
    return psi


In [5]:
def solve_particle_in_box(N=200, L=10.0, n_states=5):
    """
    Solve the particle-in-a-box problem and return results.

    Parameters
    ----------
    N : int
        Number of grid points (more points = better accuracy).
    L : float
        Box length in atomic units (Bohr radii).
    n_states : int
        Number of lowest states to analyze.

    Returns
    -------
    x : grid points
    energies : numerical eigenvalues
    wavefunctions : numerical eigenvectors (columns)
    E_exact : analytical eigenvalues
    psi_exact : analytical wavefunctions (columns)
    """
    # Build and diagonalize the Hamiltonian
    H, x, dx = build_hamiltonian(N, L)

    # numpy.linalg.eigh returns eigenvalues in ascending order, which is
    # exactly what we want (ground state first).
    energies, wavefunctions = np.linalg.eigh(H)

    # The eigenvectors from eigh are normalized such that v^T v = 1.
    # For the continuous wavefunction, we need int |psi|^2 dx = 1,
    # so psi_continuous = eigenvector / sqrt(dx).
    wavefunctions = wavefunctions / np.sqrt(dx)

    # Analytical solutions
    E_exact = analytical_energies(n_states, L)
    psi_exact = analytical_wavefunctions(x, n_states, L)

    return x, energies[:n_states], wavefunctions[:, :n_states], E_exact, psi_exact


In [6]:
def print_energy_comparison(E_numerical, E_exact):
    """Print a comparison table of numerical vs analytical energies."""
    print("=" * 65)
    print(f"{'n':>3}  {'E_numerical':>14}  {'E_analytical':>14}  {'Rel. Error':>12}")
    print("-" * 65)
    for i in range(len(E_exact)):
        rel_err = abs(E_numerical[i] - E_exact[i]) / E_exact[i]
        print(f"{i+1:>3}  {E_numerical[i]:>14.8f}  {E_exact[i]:>14.8f}  {rel_err:>12.2e}")
    print("=" * 65)


In [7]:
def plot_wavefunctions(x, psi_num, psi_exact, energies, E_exact, L):
    """
    Plot numerical and analytical wavefunctions side by side.

    Each wavefunction is offset vertically by its energy for a nice
    energy-level diagram.
    """
    n_states = psi_num.shape[1]

    fig, axes = plt.subplots(1, 2, figsize=(14, 8))

    # --- Left panel: Wavefunctions ---
    ax = axes[0]
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, n_states))

    for i in range(n_states):
        # Offset by energy for visual clarity
        offset = E_exact[i]
        scale = 0.3 * (E_exact[1] - E_exact[0])  # scale amplitude for visibility

        # Numerical (solid line)
        ax.plot(x, psi_num[:, i] * scale + offset, color=colors[i],
                linewidth=2, label=f'n={i+1} (numerical)')

        # Analytical (dashed line)
        ax.plot(x, psi_exact[:, i] * scale + offset, color=colors[i],
                linewidth=1.5, linestyle='--', alpha=0.7)

        # Energy level line
        ax.axhline(y=offset, color=colors[i], linewidth=0.5, alpha=0.3)

        # Label
        ax.text(L * 1.02, offset, f'$E_{i+1}$ = {E_exact[i]:.4f}',
                fontsize=9, va='center', color=colors[i])

    ax.set_xlabel('x (Bohr)', fontsize=12)
    ax.set_ylabel('$\\psi_n(x)$ (offset by $E_n$)', fontsize=12)
    ax.set_title('Wavefunctions: Numerical (solid) vs Analytical (dashed)', fontsize=12)
    ax.set_xlim(0, L * 1.35)

    # --- Right panel: Probability densities ---
    ax = axes[1]

    for i in range(n_states):
        offset = E_exact[i]
        scale = 0.3 * (E_exact[1] - E_exact[0])

        prob_num = psi_num[:, i]**2
        prob_exact = psi_exact[:, i]**2

        ax.fill_between(x, offset, prob_num * scale + offset,
                        color=colors[i], alpha=0.3)
        ax.plot(x, prob_num * scale + offset, color=colors[i],
                linewidth=2, label=f'n={i+1}')
        ax.plot(x, prob_exact * scale + offset, color=colors[i],
                linewidth=1.5, linestyle='--', alpha=0.7)

        ax.axhline(y=offset, color=colors[i], linewidth=0.5, alpha=0.3)

        ax.text(L * 1.02, offset, f'$E_{i+1}$',
                fontsize=9, va='center', color=colors[i])

    ax.set_xlabel('x (Bohr)', fontsize=12)
    ax.set_ylabel('$|\\psi_n(x)|^2$ (offset by $E_n$)', fontsize=12)
    ax.set_title('Probability Densities', fontsize=12)
    ax.set_xlim(0, L * 1.15)

    plt.suptitle('1D Particle in a Box -- Finite Difference Solution\n'
                 f'(N = {len(x)} grid points, L = {L} Bohr, atomic units)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('day_001_particle_in_box.png', dpi=150, bbox_inches='tight')
    plt.show()


In [8]:
def plot_convergence(L=10.0):
    """
    Study how the numerical error decreases as we increase the number of
    grid points. This demonstrates the convergence of the finite-difference
    method.
    """
    N_values = [10, 20, 50, 100, 200, 500, 1000]
    errors_E1 = []
    errors_E5 = []

    E_exact = analytical_energies(5, L)

    for N in N_values:
        H, x, dx = build_hamiltonian(N, L)
        energies = np.linalg.eigh(H)[0]
        errors_E1.append(abs(energies[0] - E_exact[0]) / E_exact[0])
        errors_E5.append(abs(energies[4] - E_exact[4]) / E_exact[4])

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.loglog(N_values, errors_E1, 'o-', linewidth=2, markersize=8, label='$E_1$ (ground state)')
    ax.loglog(N_values, errors_E5, 's-', linewidth=2, markersize=8, label='$E_5$')

    # Reference line for O(dx^2) = O(N^{-2}) convergence
    N_arr = np.array(N_values, dtype=float)
    ax.loglog(N_arr, 0.5 * (N_arr / N_arr[0])**(-2) * errors_E1[0],
              '--', color='gray', label='$O(N^{-2})$ reference')

    ax.set_xlabel('Number of grid points $N$', fontsize=12)
    ax.set_ylabel('Relative error in energy', fontsize=12)
    ax.set_title('Convergence of Finite-Difference Eigenvalues', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('day_001_convergence.png', dpi=150, bbox_inches='tight')
    plt.show()
